In [1]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
import prism


In [ ]:
# import pandas as pd
# import xarray as xr
# from imagematerials.util import dataset_to_array
# maintenance_fp = Path("../data/raw/vehicles/standard_data/maintenance_passenger_cars.csv")
# df = pd.read_csv(maintenance_fp, index_col=0)
# xr.DataArray(df["total_material_per_km"], dims=("Material",), coords={"Material": df["Material"]})
# print(df)
# dataset_to_array(df.to_xarray(), ["Material"], [])


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema2.nc")

In [3]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [ ]:
prep_data["maintenance_material_fractions"]

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

material_fractions = prep_data['material_fractions']

maintenance_material = pd.read_csv("../data/raw/vehicles/standard_data/maintenance_passenger_cars.csv", index_col=0)


# Assuming material_fractions has the needed dimensions
materials = material_fractions.coords["material"]
types = material_fractions.coords["Type"]

# Initialize xr_maintenance_material with zeros
xr_maintenance_material = xr.DataArray(
    np.zeros((len(materials), len(types))),  # Shape based on dimensions
    dims=("material", "Type"),
    coords={"material": material_fractions.coords["material"], "Type": material_fractions.coords["Type"]}
)

# Assign values from data in xr_maintenance_material where Type contains "Cars"
cars_mask = np.char.find(types.astype(str), "Cars") >= 0  # Find entries containing "Cars"
xr_maintenance_material.loc[dict(Type=types[cars_mask])] = maintenance_material["total_material_per_km"].values[:, np.newaxis]

xr_maintenance_material.sel(Type="Bikes")


In [4]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [5]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

TypeError: GenericMainModel.__init__() got an unexpected keyword argument 'compute_maintenance_materials'

In [6]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

In [7]:
main_model_factory.simulate(simulation_timeline)

In [ ]:
main_model_normal.simulate(simulation_timeline)

In [ ]:
assert(main_model_factory.stock_by_cohort_maintenance_materials == main_model_normal.maintenance_model.stock_by_cohort_maintenance_materials).all().all()

In [8]:
main_model_factory.stock_by_cohort

<xarray.DataArray (Time: 254, Cohort: 254, Region: 28, Type: 53)> Size: 766MB
array([[[[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         ...,
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

        [[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         ...,
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

        [[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         ...,
...
         ...,
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan]],

        [[nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         ...,
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan]],

        [[nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         ...,
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan]]]])
Coordinates:
  * Time     (Time) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060
  * Cohort   (Cohort) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060
  * Region   (Region) <U2 224B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
  * Type     (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'